# Notebook 05: Policy Evaluation, Explainable AI (SHAP), and Cumulative Reports
Evaluates the learned CQL policy against clinical baselines, computes SHAP feature attributions, generates natural language explanations, and builds interactive cumulative health reports.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import torch
import pandas as pd
import numpy as np
import config
from rl.state import StateConstructor
from rl.model import DiscreteCQLNetwork
from rl.evaluate import evaluate_policy
from xai.shap_explainer import RLShapExplainer
from xai.explanation_generator import generate_clinical_explanation
from report.report_generator import generate_cumulative_report, render_markdown_report, render_html_dashboard

## 1. Load Trained CQL Model and Trajectory Data

In [ ]:
traj_df = pd.read_csv(config.PROCESSED_DATA_DIR / "trajectory_data.csv")
sc = StateConstructor()

model_path = config.RL_AGENT_DIR / "cql_model.pt"
checkpoint = torch.load(model_path, map_location="cpu")
model = DiscreteCQLNetwork(state_dim=sc.state_dim)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("OK CQL Model loaded successfully.")

## 2. Policy Benchmarking: CQL vs Clinical Rule vs Behavior vs Random

In [ ]:
results = evaluate_policy(model, traj_df, state_constructor=sc)

## 3. Explainable AI: SHAP Attributions for a Specific Patient

In [ ]:
# Select a patient with a deteriorating trajectory
det_sids = traj_df[traj_df["overall_trajectory_status"] == "deteriorating"]["subject_id"].unique()
target_sid = det_sids[0]
patient_records = traj_df[traj_df["subject_id"] == target_sid].sort_values("visit_number").reset_index(drop=True)
latest_obs = patient_records.iloc[-1]

state_vec = sc.extract_state_vector(latest_obs)
action_idx = model.select_action(torch.tensor(state_vec, dtype=torch.float32))

bg_matrix = sc.extract_state_matrix(traj_df.sample(40))
explainer = RLShapExplainer(model=model, background_data=bg_matrix, state_constructor=sc)
shap_res = explainer.explain_action(state_vec, action=action_idx)

explanation = generate_clinical_explanation(
    action=action_idx,
    obs_data=latest_obs,
    attributions=shap_res["attributions"]
)

print("RECOMMENDED ACTION:", explanation["action_name"])
print("SUMMARY RATIONALE:
", explanation["summary_statement"])
print("
TOP KEY DRIVING FACTORS:")
for kf in explanation["key_factors"]:
    print(f" - {kf['clinical_description']} (SHAP: {kf['shap_attribution']:+.4f})")

## 4. Generate Cumulative Health Report

In [ ]:
from rl.actions import get_action_metadata
meta = get_action_metadata(action_idx)

rl_rec = {
    "action_code": action_idx,
    "action_name": meta["name"],
    "recommended_interval": meta["interval"],
    "urgency": meta["urgency"],
    "q_values": {config.ACTION_NAMES[i]: round(float(v), 3) for i, v in enumerate(model.get_q_values(torch.tensor(state_vec)).numpy())},
    "description": meta["description"],
}

report = generate_cumulative_report(
    patient_history_df=patient_records,
    rl_recommendation_dict=rl_rec,
    xai_explanation_dict=explanation,
)

md_content = render_markdown_report(report)
print(md_content[:1200] + "\n... [truncated] ...")